# Phase 4: Style Loss Validation
In this notebook, we test the **Gram Matrix** and the **Style Loss** to see if they correctly penalize images that do not have the same artistic textures as the Icarus painting.

In [ ]:
import sys
import os
import torch

# Add src to path
sys.path.append(os.path.abspath(os.path.join('..', 'src')))

from data.image_loader import load_image
from models.style_encoder import StyleEncoder
from training.losses import calc_style_loss

In [ ]:
# 1. Setup the GPU and the Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
style_encoder = StyleEncoder().to(device)

# 2. Load the Style Image (Icarus)
icarus = load_image('../data/style_images/icarus.jpg', max_size=400).to(device)

# 3. Load the Content Image (McGregor)
mcgregor = load_image('../data/content_images/mcgregor.jpg', max_size=400).to(device)

# 4. Create a "Noisy" Icarus
noisy_icarus = icarus + (torch.randn_like(icarus) * 0.5)
noisy_icarus = torch.clamp(noisy_icarus, 0, 1)

print("Images loaded successfully!")

In [ ]:
# 5. Extract the Gram Matrices
grams_icarus = style_encoder(icarus)
grams_mcgregor = style_encoder(mcgregor)
grams_noisy = style_encoder(noisy_icarus)

print("Gram Matrices extracted!")
print(f"There are {len(grams_icarus)} Gram Matrices (one for each style layer).")

# Notice how the Gram Matrix shape is identical, regardless of the original image's aspect ratio!
print(f"Icarus conv1_1 Gram Matrix shape: {grams_icarus['conv1_1'].shape}")
print(f"McGregor conv1_1 Gram Matrix shape: {grams_mcgregor['conv1_1'].shape}")

### The Math Test
Let's calculate the Style Loss. Because the Gram Matrix mathematically removes spatial data, we can finally compare Icarus (landscape) to McGregor (portrait) without PyTorch crashing!

In [ ]:
# Test 1: Icarus vs Icarus
loss_identical = calc_style_loss(grams_icarus, grams_icarus)
print(f"Loss (Icarus vs Icarus): {loss_identical.item():.4f}")

# Test 2: Icarus vs Noisy Icarus
loss_noisy = calc_style_loss(grams_noisy, grams_icarus)
print(f"Loss (Icarus vs Noisy Icarus): {loss_noisy.item():.4f}")

# Test 3: Icarus vs McGregor
loss_different = calc_style_loss(grams_mcgregor, grams_icarus)
print(f"Loss (Icarus vs McGregor): {loss_different.item():.4f}")